# 배터리 슈레더 발화/열폭주 감지 시스템

## 프로젝트 개요
- **목적**: 배터리 슈레더 운용 중 발화 및 열폭주(Thermal Runaway) 실시간 감지
- **센서 구성**: IR 비접촉 온도센서 2채널 (표면) + 접촉식 온도센서 2채널 (하우징)
- **데이터**: 600초 (10분), 100ms 간격, 총 6,000 샘플
- **감지 방식**: 온도변화율(dT/dt) + 듀얼 IR 교차검증 + 누적확률 모델

## 시나리오 구성 (5단계)
| 구간 | 시간(초) | 상태 | 설명 |
|------|----------|------|------|
| Phase 1 | 0 ~ 120 | 정상 운전 | 안정적 온도 유지 |
| Phase 2 | 120 ~ 200 | 전조 징후 | 미세 온도 상승, dT/dt 증가 |
| Phase 3 | 200 ~ 350 | 발화 | 급격한 온도 상승 |
| Phase 4 | 350 ~ 500 | 열폭주 | 최고 온도 도달, 연쇄 반응 |
| Phase 5 | 500 ~ 600 | 냉각 | 소화 시스템 작동 후 온도 하강 |

## Step 0. 라이브러리 임포트

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings('ignore')

# 재현성을 위한 시드 고정
np.random.seed(42)

print("라이브러리 로드 완료")
print(f"  - NumPy: {np.__version__}")
print(f"  - Pandas: {pd.__version__}")

## Step 1. 발화 시나리오 데이터 생성

5단계 시나리오에 따른 4채널 온도 데이터를 생성합니다.

- **IR1, IR2**: 비접촉 적외선 센서 (슈레더 표면 온도, 정상 ~45°C, 발화 시 최대 300°C)
- **TMP_A, TMP_B**: 접촉식 온도센서 (하우징 온도, 정상 ~35°C, 발화 시 최대 150°C)
- IR 센서는 빠른 응답, TMP 센서는 열전달 지연 반영

In [ ]:
# 시간축 생성
dt = 0.1  # 100ms 간격
total_time = 600  # 600초 = 10분
N = int(total_time / dt)  # 6,000 샘플
t = np.linspace(0, total_time, N)

# 페이즈 경계 정의
phases = {
    'Normal':          (0, 120),
    'Precursor':       (120, 200),
    'Fire':            (200, 350),
    'Thermal Runaway': (350, 500),
    'Cooling':         (500, 600)
}

# 페이즈 인덱스 매핑
phase_labels = np.empty(N, dtype='U20')
for name, (t_start, t_end) in phases.items():
    mask = (t >= t_start) & (t < t_end)
    phase_labels[mask] = name
phase_labels[t >= 600] = 'Cooling'

def generate_temperature_profile(t, sensor_type='IR', offset=0):
    temp = np.zeros_like(t)
    for i, ti in enumerate(t):
        if ti < 120:
            if sensor_type == 'IR':
                temp[i] = 45.0 + offset + np.sin(ti * 0.05) * 2
            else:
                temp[i] = 35.0 + offset + np.sin(ti * 0.03) * 1.5
        elif ti < 200:
            progress = (ti - 120) / 80
            if sensor_type == 'IR':
                base = 45.0 + offset
                temp[i] = base + progress**2 * 30 + np.sin(ti * 0.1) * 3
            else:
                delayed_progress = max(0, (ti - 130) / 80)
                base = 35.0 + offset
                temp[i] = base + delayed_progress**2 * 15 + np.sin(ti * 0.08) * 2
        elif ti < 350:
            progress = (ti - 200) / 150
            if sensor_type == 'IR':
                temp[i] = 75.0 + offset + progress * 180 + np.sin(ti * 0.2) * 5
            else:
                delayed_progress = min(1, max(0, (ti - 215) / 150))
                temp[i] = 50.0 + offset + delayed_progress * 80 + np.sin(ti * 0.15) * 3
        elif ti < 500:
            progress = (ti - 350) / 150
            if sensor_type == 'IR':
                peak = 280 + offset
                temp[i] = peak + np.sin(progress * np.pi) * 20 + np.random.randn() * 8
            else:
                peak = 140 + offset
                temp[i] = peak + np.sin(progress * np.pi) * 10 + np.random.randn() * 4
        else:
            progress = (ti - 500) / 100
            if sensor_type == 'IR':
                start_temp = 280 + offset
                target = 60 + offset
                temp[i] = target + (start_temp - target) * np.exp(-progress * 2.5)
            else:
                start_temp = 140 + offset
                target = 45 + offset
                temp[i] = target + (start_temp - target) * np.exp(-progress * 1.8)

    noise_level = 0.5 if sensor_type == 'IR' else 0.3
    temp += np.random.randn(N) * noise_level
    return temp

IR1 = generate_temperature_profile(t, 'IR', offset=0)
IR2 = generate_temperature_profile(t, 'IR', offset=2)
TMP_A = generate_temperature_profile(t, 'TMP', offset=0)
TMP_B = generate_temperature_profile(t, 'TMP', offset=1)

df = pd.DataFrame({
    'time': t, 'IR1': IR1, 'IR2': IR2,
    'TMP_A': TMP_A, 'TMP_B': TMP_B, 'phase': phase_labels
})

print("데이터 생성 완료")
print(f"  - 총 샘플 수: {len(df):,}")
print(f"  - 시간 범위: {df['time'].min():.1f} ~ {df['time'].max():.1f} 초")
print(f"  - 샘플링 간격: {dt*1000:.0f} ms")
print(f"\n페이즈별 샘플 수:")
for phase_name in phases:
    count = (df['phase'] == phase_name).sum()
    print(f"  {phase_name:20s}: {count:,} 샘플")
print(f"\n온도 범위:")
for col in ['IR1', 'IR2', 'TMP_A', 'TMP_B']:
    print(f"  {col:6s}: {df[col].min():6.1f} ~ {df[col].max():6.1f} °C")
df.head(10)

## Step 2. 데이터 시각화 (전체 온도 프로파일)

4채널 온도 데이터와 페이즈 경계를 확인합니다.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

phase_colors = {
    'Normal': '#e8f5e9', 'Precursor': '#fff9c4',
    'Fire': '#ffccbc', 'Thermal Runaway': '#f8bbd0', 'Cooling': '#e3f2fd'
}
for name, (ts, te) in phases.items():
    ax.axvspan(ts, te, alpha=0.3, color=phase_colors[name], label=name)

ax.plot(df['time'], df['IR1'], color='#d32f2f', linewidth=0.8, label='IR1 (Surface)')
ax.plot(df['time'], df['IR2'], color='#ff7043', linewidth=0.8, label='IR2 (Surface)')
ax.plot(df['time'], df['TMP_A'], color='#1565c0', linewidth=0.8, label='TMP_A (Housing)')
ax.plot(df['time'], df['TMP_B'], color='#42a5f5', linewidth=0.8, label='TMP_B (Housing)')

for name, (ts, te) in phases.items():
    if ts > 0:
        ax.axvline(ts, color='gray', linestyle='--', alpha=0.5, linewidth=0.8)

ax.set_xlabel('Time (seconds)', fontsize=12)
ax.set_ylabel('Temperature (deg C)', fontsize=12)
ax.set_title('Battery Shredder - 4 Channel Temperature Profile', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=9, ncol=3)
ax.set_xlim(0, 600)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 3. 발화 감지 알고리즘 설계

### 감지 엔진 3중 구조:
1. **온도변화율 (dT/dt)**: 이동 윈도우 기반 미분, 임계값 기반 경보
   - dT/dt > 2°C/s: 전조 (Precursor)
   - dT/dt > 10°C/s: 발화 (Fire)
2. **듀얼 IR 교차검증**: IR1과 IR2 모두 동시에 임계값 초과 시에만 확정
3. **누적확률 모델**: P = 1 - exp(-Sigma evidence), 증거가 쌓일수록 확률 증가

### 경보 레벨:
- **Normal (0)**: P < 0.3, 정상 운전
- **Warning (1)**: 0.3 <= P < 0.7, 주의 필요
- **Emergency (2)**: P >= 0.7, 즉시 정지 및 소화

In [ ]:
class FireDetector:
    def __init__(self, dt=0.1, window_size=10):
        self.dt = dt
        self.window_size = window_size
        self.dTdt_precursor = 2.0
        self.dTdt_fire = 10.0
        self.ir_abs_threshold = 80.0
        self.tmp_abs_threshold = 55.0
        self.evidence_decay = 0.98
        self.warning_threshold = 0.3
        self.emergency_threshold = 0.7

    def compute_rate_of_change(self, signal):
        dTdt = np.zeros_like(signal)
        for i in range(self.window_size, len(signal)):
            dTdt[i] = (signal[i] - signal[i - self.window_size]) / (self.window_size * self.dt)
        return dTdt

    def detect(self, df):
        n = len(df)
        dTdt_IR1 = self.compute_rate_of_change(df['IR1'].values)
        dTdt_IR2 = self.compute_rate_of_change(df['IR2'].values)
        dTdt_TMP_A = self.compute_rate_of_change(df['TMP_A'].values)
        dTdt_TMP_B = self.compute_rate_of_change(df['TMP_B'].values)

        dTdt_IR_max = np.maximum(np.abs(dTdt_IR1), np.abs(dTdt_IR2))
        dTdt_TMP_max = np.maximum(np.abs(dTdt_TMP_A), np.abs(dTdt_TMP_B))

        evidence = np.zeros(n)
        cumulative_evidence = np.zeros(n)
        fire_probability = np.zeros(n)
        alert_level = np.zeros(n, dtype=int)

        for i in range(1, n):
            e = 0.0

            if dTdt_IR_max[i] > self.dTdt_fire:
                e += 0.5
            elif dTdt_IR_max[i] > self.dTdt_precursor:
                e += 0.15

            if dTdt_TMP_max[i] > self.dTdt_precursor:
                e += 0.1

            ir1_high = df['IR1'].values[i] > self.ir_abs_threshold
            ir2_high = df['IR2'].values[i] > self.ir_abs_threshold
            if ir1_high and ir2_high:
                e += 0.3
            elif ir1_high or ir2_high:
                e += 0.05

            if df['TMP_A'].values[i] > self.tmp_abs_threshold and df['TMP_B'].values[i] > self.tmp_abs_threshold:
                e += 0.15

            evidence[i] = e
            cumulative_evidence[i] = cumulative_evidence[i-1] * self.evidence_decay + e
            fire_probability[i] = 1.0 - np.exp(-cumulative_evidence[i])

            if fire_probability[i] >= self.emergency_threshold:
                alert_level[i] = 2
            elif fire_probability[i] >= self.warning_threshold:
                alert_level[i] = 1

        return pd.DataFrame({
            'time': df['time'].values,
            'dTdt_IR1': dTdt_IR1, 'dTdt_IR2': dTdt_IR2,
            'dTdt_TMP_A': dTdt_TMP_A, 'dTdt_TMP_B': dTdt_TMP_B,
            'dTdt_IR_max': dTdt_IR_max,
            'evidence': evidence, 'cumulative_evidence': cumulative_evidence,
            'fire_probability': fire_probability,
            'alert_level': alert_level, 'phase': df['phase'].values
        })

detector = FireDetector(dt=0.1, window_size=10)
print("발화 감지 엔진 초기화 완료")
print(f"  - dT/dt 윈도우: {detector.window_size} 샘플 ({detector.window_size * detector.dt:.1f}초)")
print(f"  - 전조 임계값: dT/dt > {detector.dTdt_precursor} deg C/s")
print(f"  - 발화 임계값: dT/dt > {detector.dTdt_fire} deg C/s")
print(f"  - IR 절대 임계값: > {detector.ir_abs_threshold} deg C")
print(f"  - 경보: Normal(P<{detector.warning_threshold}) / Warning / Emergency(P>={detector.emergency_threshold})")

## Step 4. 감지 실행

6,000개 전체 샘플에 대해 발화 감지를 수행하고, 감지 시점 및 지연시간을 기록합니다.

In [ ]:
results = detector.detect(df)

warning_mask = results['alert_level'] >= 1
first_warning_idx = np.argmax(warning_mask) if warning_mask.any() else None
first_warning_time = results['time'].iloc[first_warning_idx] if first_warning_idx else None

emergency_mask = results['alert_level'] >= 2
first_emergency_idx = np.argmax(emergency_mask) if emergency_mask.any() else None
first_emergency_time = results['time'].iloc[first_emergency_idx] if first_emergency_idx else None

actual_fire_start = 200.0
warning_latency = first_warning_time - actual_fire_start if first_warning_time else None
emergency_latency = first_emergency_time - actual_fire_start if first_emergency_time else None

normal_phase_mask = results['phase'] == 'Normal'
false_alarms_normal = (results.loc[normal_phase_mask, 'alert_level'] > 0).sum()

print("=" * 60)
print("  감지 실행 결과")
print("=" * 60)
print(f"\n  총 처리 샘플: {len(results):,}")
print(f"\n  [감지 시점]")
print(f"    실제 발화 시작:     {actual_fire_start:.1f} 초")
if first_warning_time:
    print(f"    첫 Warning 감지:    {first_warning_time:.1f} 초 (지연: {warning_latency:+.1f} 초)")
if first_emergency_time:
    print(f"    첫 Emergency 감지:  {first_emergency_time:.1f} 초 (지연: {emergency_latency:+.1f} 초)")

print(f"\n  [경보 통계]")
for level, name in [(0, 'Normal'), (1, 'Warning'), (2, 'Emergency')]:
    count = (results['alert_level'] == level).sum()
    pct = count / len(results) * 100
    print(f"    {name:12s}: {count:5,} 샘플 ({pct:5.1f}%)")

print(f"\n  [오경보 분석]")
print(f"    정상 구간 오경보: {false_alarms_normal} 건")
print("=" * 60)

## Step 5. 결과 분석

페이즈별 감지 성능을 상세 분석합니다.

In [ ]:
print("페이즈별 감지 성능 분석")
print("=" * 80)
print(f"{'Phase':20s} | {'Samples':>8s} | {'Avg P':>11s} | {'Max P':>11s} | {'Warning%':>8s} | {'Emergency%':>10s}")
print("-" * 80)

for phase_name in phases:
    mask = results['phase'] == phase_name
    subset = results[mask]
    n_samples = len(subset)
    avg_prob = subset['fire_probability'].mean()
    max_prob = subset['fire_probability'].max()
    w_pct = (subset['alert_level'] >= 1).mean() * 100
    e_pct = (subset['alert_level'] >= 2).mean() * 100
    print(f"{phase_name:20s} | {n_samples:8,} | {avg_prob:11.4f} | {max_prob:11.4f} | {w_pct:7.1f}% | {e_pct:9.1f}%")

print("=" * 80)

print(f"\n온도변화율(dT/dt) 통계 (deg C/s)")
print("-" * 60)
print(f"{'Phase':20s} | {'IR max':>10s} | {'TMP_A max':>10s}")
print("-" * 60)
for phase_name in phases:
    mask = results['phase'] == phase_name
    ir_max = results.loc[mask, 'dTdt_IR_max'].max()
    tmp_max = results.loc[mask, 'dTdt_TMP_A'].abs().max()
    print(f"{phase_name:20s} | {ir_max:10.2f} | {tmp_max:10.2f}")

precursor_mask = results['phase'] == 'Precursor'
precursor_warnings = (results.loc[precursor_mask, 'alert_level'] >= 1).any()
print(f"\n조기 감지 (전조 단계에서 Warning): {'성공' if precursor_warnings else '실패'}")
if precursor_warnings:
    pw = results.loc[precursor_mask & (results['alert_level'] >= 1), 'time'].iloc[0]
    print(f"  전조 단계 첫 Warning: {pw:.1f}초 (발화 {200 - pw:.1f}초 전 감지)")

## Step 6. 시각화

각 분석 결과를 개별 차트로 시각화합니다.

### 6-1. 4채널 온도 시계열 (페이즈 컬러)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

phase_colors = {
    'Normal': '#c8e6c9', 'Precursor': '#fff9c4',
    'Fire': '#ffccbc', 'Thermal Runaway': '#f8bbd0', 'Cooling': '#bbdefb'
}

for ax in [ax1, ax2]:
    for name, (ts, te) in phases.items():
        ax.axvspan(ts, te, alpha=0.4, color=phase_colors[name])
    for name, (ts, te) in phases.items():
        if ts > 0:
            ax.axvline(ts, color='gray', linestyle='--', alpha=0.4, linewidth=0.7)

ax1.plot(df['time'], df['IR1'], color='#c62828', linewidth=0.7, label='IR1 (Surface)')
ax1.plot(df['time'], df['IR2'], color='#e65100', linewidth=0.7, label='IR2 (Surface)')
ax1.axhline(detector.ir_abs_threshold, color='red', linestyle=':', alpha=0.6,
            label=f'IR Threshold ({detector.ir_abs_threshold} deg C)')
ax1.set_ylabel('Temperature (deg C)', fontsize=11)
ax1.set_title('IR Sensors - Surface Temperature', fontsize=13, fontweight='bold')
ax1.legend(loc='upper left', fontsize=9)
ax1.grid(True, alpha=0.3)

ax2.plot(df['time'], df['TMP_A'], color='#0d47a1', linewidth=0.7, label='TMP_A (Housing)')
ax2.plot(df['time'], df['TMP_B'], color='#1976d2', linewidth=0.7, label='TMP_B (Housing)')
ax2.axhline(detector.tmp_abs_threshold, color='blue', linestyle=':', alpha=0.6,
            label=f'TMP Threshold ({detector.tmp_abs_threshold} deg C)')
ax2.set_xlabel('Time (seconds)', fontsize=11)
ax2.set_ylabel('Temperature (deg C)', fontsize=11)
ax2.set_title('TMP Sensors - Housing Temperature', fontsize=13, fontweight='bold')
ax2.legend(loc='upper left', fontsize=9)
ax2.grid(True, alpha=0.3)

for name, (ts, te) in phases.items():
    mid = (ts + te) / 2
    ax1.text(mid, ax1.get_ylim()[1] * 0.95, name, ha='center', va='top',
             fontsize=8, fontweight='bold', alpha=0.7)

plt.tight_layout()
plt.show()

### 6-2. 온도변화율 (dT/dt) + 임계값 라인

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

for name, (ts, te) in phases.items():
    ax.axvspan(ts, te, alpha=0.2, color=phase_colors[name])

ax.plot(results['time'], results['dTdt_IR1'], color='#c62828', linewidth=0.6, alpha=0.7, label='dT/dt IR1')
ax.plot(results['time'], results['dTdt_IR2'], color='#e65100', linewidth=0.6, alpha=0.7, label='dT/dt IR2')
ax.plot(results['time'], results['dTdt_TMP_A'], color='#0d47a1', linewidth=0.6, alpha=0.7, label='dT/dt TMP_A')

ax.axhline(detector.dTdt_precursor, color='orange', linestyle='--', linewidth=1.5,
           label=f'Precursor ({detector.dTdt_precursor} deg C/s)')
ax.axhline(detector.dTdt_fire, color='red', linestyle='--', linewidth=1.5,
           label=f'Fire ({detector.dTdt_fire} deg C/s)')
ax.axhline(-detector.dTdt_precursor, color='orange', linestyle='--', linewidth=1.0, alpha=0.5)
ax.axhline(-detector.dTdt_fire, color='red', linestyle='--', linewidth=1.0, alpha=0.5)

ax.set_xlabel('Time (seconds)', fontsize=11)
ax.set_ylabel('Rate of Change (deg C/s)', fontsize=11)
ax.set_title('Temperature Rate of Change (dT/dt) - All Channels', fontsize=13, fontweight='bold')
ax.legend(loc='upper left', fontsize=9)
ax.set_xlim(0, 600)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 6-3. 발화 확률 누적 곡선

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

for name, (ts, te) in phases.items():
    ax.axvspan(ts, te, alpha=0.2, color=phase_colors[name])

ax.fill_between(results['time'], results['fire_probability'], alpha=0.3, color='red')
ax.plot(results['time'], results['fire_probability'], color='#c62828', linewidth=1.2, label='Fire Probability')

ax.axhline(detector.warning_threshold, color='orange', linestyle='--', linewidth=1.5,
           label=f'Warning Threshold ({detector.warning_threshold})')
ax.axhline(detector.emergency_threshold, color='red', linestyle='--', linewidth=1.5,
           label=f'Emergency Threshold ({detector.emergency_threshold})')

if first_warning_time:
    ax.axvline(first_warning_time, color='orange', linestyle=':', alpha=0.8)
    ax.annotate(f'First Warning\n{first_warning_time:.1f}s',
                xy=(first_warning_time, detector.warning_threshold),
                xytext=(first_warning_time + 20, 0.5),
                arrowprops=dict(arrowstyle='->', color='orange'),
                fontsize=9, color='orange', fontweight='bold')

if first_emergency_time:
    ax.axvline(first_emergency_time, color='red', linestyle=':', alpha=0.8)
    ax.annotate(f'First Emergency\n{first_emergency_time:.1f}s',
                xy=(first_emergency_time, detector.emergency_threshold),
                xytext=(first_emergency_time + 20, 0.85),
                arrowprops=dict(arrowstyle='->', color='red'),
                fontsize=9, color='red', fontweight='bold')

ax.set_xlabel('Time (seconds)', fontsize=11)
ax.set_ylabel('Fire Probability', fontsize=11)
ax.set_title('Cumulative Fire Probability - P = 1 - exp(-cumulative evidence)', fontsize=13, fontweight='bold')
ax.legend(loc='center left', fontsize=9)
ax.set_xlim(0, 600)
ax.set_ylim(-0.05, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 6-4. 경보 레벨 타임라인

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

alert_colors_map = {0: '#4caf50', 1: '#ff9800', 2: '#f44336'}
alert_names = {0: 'Normal', 1: 'Warning', 2: 'Emergency'}

for level in [0, 1, 2]:
    mask = results['alert_level'] == level
    times = results['time'][mask].values
    if len(times) > 0:
        ax.scatter(times, results['alert_level'][mask].values,
                  c=alert_colors_map[level], s=1, alpha=0.6, label=alert_names[level])

for name, (ts, te) in phases.items():
    if ts > 0:
        ax.axvline(ts, color='gray', linestyle='--', alpha=0.4, linewidth=0.7)
        ax.text(ts, 2.3, name, ha='left', va='bottom', fontsize=8, alpha=0.6)

ax.set_xlabel('Time (seconds)', fontsize=11)
ax.set_ylabel('Alert Level', fontsize=11)
ax.set_title('Alert Level Timeline', fontsize=13, fontweight='bold')
ax.set_yticks([0, 1, 2])
ax.set_yticklabels(['Normal (0)', 'Warning (1)', 'Emergency (2)'])
ax.set_xlim(0, 600)
ax.set_ylim(-0.3, 2.8)
ax.legend(loc='upper left', fontsize=9, markerscale=8)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

### 6-5. IR1 vs IR2 교차검증 산점도

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

phase_scatter_colors = {
    'Normal': '#4caf50', 'Precursor': '#ffc107',
    'Fire': '#ff5722', 'Thermal Runaway': '#e91e63', 'Cooling': '#2196f3'
}

for phase_name in phases:
    mask = df['phase'] == phase_name
    ax.scatter(df.loc[mask, 'IR1'], df.loc[mask, 'IR2'],
              c=phase_scatter_colors[phase_name], s=3, alpha=0.4, label=phase_name)

ax.axvline(detector.ir_abs_threshold, color='red', linestyle='--', alpha=0.6, linewidth=1)
ax.axhline(detector.ir_abs_threshold, color='red', linestyle='--', alpha=0.6, linewidth=1)

ax.fill_between([detector.ir_abs_threshold, 350], detector.ir_abs_threshold, 350,
                alpha=0.1, color='red', label='Cross-validated Fire Zone')

lims = [min(df['IR1'].min(), df['IR2'].min()), max(df['IR1'].max(), df['IR2'].max())]
ax.plot(lims, lims, 'k--', alpha=0.3, linewidth=0.8, label='Perfect Correlation')

ax.set_xlabel('IR1 Temperature (deg C)', fontsize=11)
ax.set_ylabel('IR2 Temperature (deg C)', fontsize=11)
ax.set_title('IR1 vs IR2 Cross-Validation Scatter', fontsize=13, fontweight='bold')
ax.legend(loc='upper left', fontsize=9, markerscale=4)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 6-6. 감지 지연시간 요약

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
events, event_times, event_colors = [], [], []
events.append('Actual Fire Start'); event_times.append(actual_fire_start); event_colors.append('#9e9e9e')
if first_warning_time:
    events.append('First Warning'); event_times.append(first_warning_time); event_colors.append('#ff9800')
if first_emergency_time:
    events.append('First Emergency'); event_times.append(first_emergency_time); event_colors.append('#f44336')
events.append('Thermal Runaway'); event_times.append(350.0); event_colors.append('#e91e63')

bars = ax1.barh(events, event_times, color=event_colors, edgecolor='white', height=0.6)
for bar, t_val in zip(bars, event_times):
    ax1.text(t_val + 5, bar.get_y() + bar.get_height()/2,
             f'{t_val:.1f}s', va='center', fontsize=10, fontweight='bold')
ax1.set_xlabel('Time (seconds)', fontsize=11)
ax1.set_title('Event Detection Timeline', fontsize=13, fontweight='bold')
ax1.set_xlim(0, max(event_times) * 1.3)
ax1.grid(True, alpha=0.3, axis='x')

ax2 = axes[1]
lat_labels, lat_vals, lat_colors = [], [], []
if warning_latency is not None:
    lat_labels.append('Warning\nLatency'); lat_vals.append(warning_latency); lat_colors.append('#ff9800')
if emergency_latency is not None:
    lat_labels.append('Emergency\nLatency'); lat_vals.append(emergency_latency); lat_colors.append('#f44336')
if first_emergency_time and first_emergency_time < 350:
    sm = 350 - first_emergency_time
    lat_labels.append('Safety Margin\n(before T.R.)'); lat_vals.append(sm); lat_colors.append('#4caf50')

bars2 = ax2.bar(lat_labels, lat_vals, color=lat_colors, edgecolor='white', width=0.5)
for bar, val in zip(bars2, lat_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{val:.1f}s', ha='center', fontsize=11, fontweight='bold')
ax2.set_ylabel('Time (seconds)', fontsize=11)
ax2.set_title('Detection Latency Analysis', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Step 7. 종합 요약

### 감지 성능 메트릭 및 안전 마진 분석

In [ ]:
print("=" * 70)
print("  배터리 슈레더 발화/열폭주 감지 시스템 - 종합 평가 보고서")
print("=" * 70)

print("\n[1] 시스템 구성")
print(f"    센서: IR1, IR2 (비접촉 표면), TMP_A, TMP_B (접촉 하우징)")
print(f"    샘플링: {dt*1000:.0f}ms 간격, 총 {N:,} 샘플 ({total_time}초)")
print(f"    감지 방식: dT/dt + 듀얼 IR 교차검증 + 누적확률 모델")

print("\n[2] 감지 성능")
print(f"    실제 발화 시작:       {actual_fire_start:.1f}초")
if first_warning_time:
    if warning_latency < 0:
        print(f"    Warning 감지:         {first_warning_time:.1f}초 (발화 {abs(warning_latency):.1f}초 전 조기감지)")
    else:
        print(f"    Warning 감지:         {first_warning_time:.1f}초 (발화 후 {warning_latency:.1f}초)")
if first_emergency_time:
    print(f"    Emergency 감지:       {first_emergency_time:.1f}초 (발화 후 {emergency_latency:.1f}초)")
    print(f"    열폭주 전 안전마진:   {350 - first_emergency_time:.1f}초")

print("\n[3] 오경보 분석")
print(f"    정상 구간 오경보:     {false_alarms_normal} 건 / {(df['phase']=='Normal').sum():,} 샘플")
false_alarm_rate = false_alarms_normal / (df['phase']=='Normal').sum() * 100
print(f"    오경보율:             {false_alarm_rate:.4f}%")

print("\n[4] 안전 마진 평가")
if first_emergency_time and first_emergency_time < 350:
    margin = 350 - first_emergency_time
    if margin > 60:
        grade = "우수 (60초 이상 여유)"
    elif margin > 30:
        grade = "양호 (30초 이상 여유)"
    else:
        grade = "주의 (30초 미만)"
    print(f"    열폭주 전 Emergency 감지: {margin:.1f}초 여유 -> {grade}")
else:
    print(f"    열폭주 전 Emergency 감지: 실패 (열폭주 후 감지)")

print("\n[5] 종합 판정")
if false_alarms_normal == 0 and first_emergency_time and first_emergency_time < 350:
    print("    - 오경보 없이 열폭주 전 감지 성공")
    print("    - 소화 시스템 작동 충분한 시간 확보")
    print("    -> 판정: 실전 배치 적합")
else:
    print("    -> 파라미터 튜닝 필요")

print("\n" + "=" * 70)